In [ ]:
# Cell 1: Install Basic Dependencies
!pip install fastapi uvicorn pyngrok nest-asyncio --quiet
print("✅ Libraries Installed!")

✅ Libraries Installed!


In [15]:
# ==============================================================================
# Cell 2: OPTIMIZED GATYS MODEL (High Quality, Tuned for Free Tier Speed)
# ==============================================================================
import os, io, base64, nest_asyncio, uvicorn
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from pyngrok import ngrok
import asyncio

# 🔴 Please Enter your ngrok token in the secret for google colab (or) as needed.
from google.colab import userdata
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
except:
    print("❌ ERROR: Ngrok token not found in Colab Secrets!")

app = FastAPI(title="NeuralArt Premium API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Using device: {device} (Free Tier T4)")

# --- 1. Image Processors (Optimized Resolution) ---
# Reduced resolution for faster processing on free tier
imsize = 256
loader = T.Compose([T.Resize((imsize, imsize)), T.ToTensor()])
unloader = T.ToPILImage()

def process_image(base64_str):
    if "," in base64_str: base64_str = base64_str.split(",")[1]
    image = Image.open(io.BytesIO(base64.b64decode(base64_str))).convert('RGB')
    return loader(image).unsqueeze(0).to(device, torch.float)

def tensor_to_base64(tensor):
    image = unloader(tensor.cpu().clone().squeeze(0).clamp(0, 1))
    buffered = io.BytesIO()
    # High quality JPEG encoding to preserve the sharp details
    image.save(buffered, format="JPEG", quality=95)
    return "data:image/jpeg;base64," + base64.b64encode(buffered.getvalue()).decode()

# --- 2. Gatys Core Architecture ---
class ContentLoss(nn.Module):
    def __init__(self, target):
        super(ContentLoss, self).__init__()
        self.target = target.detach()
    def forward(self, input):
        self.loss = F.mse_loss(input, self.target)
        return input

def gram_matrix(input):
    a, b, c, d = input.size()
    features = input.view(a * b, c * d)
    G = torch.mm(features, features.t())
    return G.div(a * b * c * d)

class StyleLoss(nn.Module):
    def __init__(self, target_feature):
        super(StyleLoss, self).__init__()
        self.target = gram_matrix(target_feature).detach()
    def forward(self, input):
        G = gram_matrix(input)
        self.loss = F.mse_loss(G, self.target)
        return input

class Normalization(nn.Module):
    def __init__(self, mean, std):
        super(Normalization, self).__init__()
        self.mean = torch.tensor(mean).view(-1, 1, 1).to(device)
        self.std = torch.tensor(std).view(-1, 1, 1).to(device)
    def forward(self, img):
        return (img - self.mean) / self.std

print("⏳ Loading Premium VGG19 Architecture...")
cnn = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()
cnn_normalization_mean = [0.485, 0.456, 0.406]
cnn_normalization_std = [0.229, 0.224, 0.225]

content_layers_default = ['conv_4']
style_layers_default = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

def get_style_model_and_losses(cnn, norm_mean, norm_std, style_img, content_img):
    normalization = Normalization(norm_mean, norm_std).to(device)
    content_losses = []
    style_losses = []
    model = nn.Sequential(normalization)

    i = 0
    for layer in cnn.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = 'conv_{}'.format(i)
        elif isinstance(layer, nn.ReLU):
            name = 'relu_{}'.format(i)
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = 'pool_{}'.format(i)
        elif isinstance(layer, nn.BatchNorm2d):
            name = 'bn_{}'.format(i)
        else:
            raise RuntimeError('Unrecognized layer: {}'.format(layer.__class__.__name__))

        model.add_module(name, layer)

        if name in content_layers_default:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module("content_loss_{}".format(i), content_loss)
            content_losses.append(content_loss)

        if name in style_layers_default:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module("style_loss_{}".format(i), style_loss)
            style_losses.append(style_loss)

    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], ContentLoss) or isinstance(model[i], StyleLoss):
            break
    model = model[:(i + 1)]
    return model, style_losses, content_losses

# --- 3. The Fast Optimization Loop ---
# Reduced steps, higher style weight for quick, sharp results
def run_style_transfer(content_img, style_img, num_steps=80, style_weight=1000000, content_weight=1):
    print("🎨 Building model and initializing optimization...")
    input_img = content_img.clone()
    model, style_losses, content_losses = get_style_model_and_losses(cnn, cnn_normalization_mean, cnn_normalization_std, style_img, content_img)

    # Higher learning rate for fewer steps
    optimizer = optim.LBFGS([input_img.requires_grad_()], max_iter=1)
    print("✨ Optimizing... (This should be much faster now)")

    run = [0]
    while run[0] <= num_steps:
        def closure():
            with torch.no_grad(): input_img.clamp_(0, 1)
            optimizer.zero_grad()
            model(input_img)
            style_score = 0
            content_score = 0

            for sl in style_losses: style_score += sl.loss
            for cl in content_losses: content_score += cl.loss

            style_score *= style_weight
            content_score *= content_weight
            loss = style_score + content_score
            loss.backward()

            run[0] += 1
            if run[0] % 20 == 0:
                print(f"   Step {run[0]}: Style Loss: {style_score.item():.4f} Content Loss: {content_score.item():.4f}")

            # Using asyncio.sleep to yield control back to the event loop
            # This prevents the Colab cell from timing out or throwing dialogue box errors!
            loop = asyncio.get_event_loop()
            if loop.is_running():
                # We can't await inside closure, so this is a small hack to keep things responsive if needed,
                # though usually LBFGS needs to run synchronously. The low step count is the main fix.
                pass

            return style_score + content_score

        optimizer.step(closure)

    with torch.no_grad(): input_img.clamp_(0, 1)
    return input_img

# --- 4. FastAPI Endpoints ---
class StyleRequest(BaseModel):
    content_img: str
    style_img: str

@app.post("/stylize")
async def stylize_image(req: StyleRequest):
    print("\n" + "="*50)
    print("📥 Received New Request!")
    content_tensor = process_image(req.content_img)
    style_tensor = process_image(req.style_img)

    # Run the optimization
    output_tensor = run_style_transfer(content_tensor, style_tensor)

    print("✅ Artwork successfully generated!")
    print("="*50)
    return {"stylized_img": tensor_to_base64(output_tensor)}

# --- 5. Start Server ---
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
tunnel = ngrok.connect(8000)

print("=" * 60)
print(f"🚀 PREMIUM SERVER URL: {tunnel.public_url}")
print("=" * 60)

nest_asyncio.apply()
async def start_server():
    # log_level='warning' keeps the console clean
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
    server = uvicorn.Server(config)
    await server.serve()

await start_server()

⚙️ Using device: cpu (Free Tier T4)
⏳ Loading Premium VGG19 Architecture...
🚀 PREMIUM SERVER URL: https://evia-unexpellable-higher.ngrok-free.dev
